In [ ]:
# ルートに移動

%cd ..

In [ ]:
# ライブラリのインポート

import glob
import os

import cv2
import numpy as np
from ultralytics.utils.ops import xywhn2xyxy
from ultralytics.utils.plotting import Annotator, colors

In [ ]:
# ディレクトリの定義

annotated_dir = os.path.join("data", "annotated")
prepared_dir = os.path.join("data", "prepared")

In [ ]:
# アノテーションの実行

os.makedirs(annotated_dir, exist_ok=True)

labels = sorted(
    glob.glob(os.path.join(prepared_dir, "labels", "train", "T*.txt")),
    key=lambda p: int(os.path.basename(p)[1:-4]),
)

images = sorted(
    glob.glob(os.path.join(prepared_dir, "images", "train", "T*.jpg")),
    key=lambda p: int(os.path.basename(p)[1:-4]),
)

for label, image in zip(labels, images):
    image_bgr = cv2.imread(image)
    height, width = image_bgr.shape[:2]
    annotator = Annotator(image_bgr, line_width=2)

    result = np.loadtxt(label, dtype=np.float32, ndmin=2)
    xyxy = xywhn2xyxy(result[:, 1:5].astype(np.float32), w=width, h=height)
    for class_id, x, y, w, h in result:
        xyxy = xywhn2xyxy(
            np.array([[x, y, w, h]], dtype=np.float32), w=width, h=height
        )[0]
        category_id = int(class_id) + 1
        annotator.box_label(xyxy, label=str(category_id), color=colors(category_id))

    file_name = os.path.basename(image)

    save_path = os.path.join(annotated_dir, file_name)
    cv2.imwrite(save_path, annotator.result())